# Predictive Meta-Modeling

This notebook is a direct continuation of `01_mfa.ipynb`. It loads the dataset-level, preprocessed, redundancy-reduced matrices exported by `01_mfa.ipynb` and asks a separate question from the association analysis:

> Can the reduced meta-feature matrices predict the dataset-level performance gap out of sample?

This is an exploratory predictive analysis, not a causal or confirmatory inference step.

## Minimal Predictive Protocol

We evaluate the regression/classification matrix and the classification-augmented matrix separately. Each row is one dataset. The target is `delta_norm`.

For each matrix, we compare a mean-only baseline with ridge regression under leave-one-dataset-out cross-validation. We additionally include a shallow decision tree and, when `xgboost` is installed, a strongly regularized XGBoost model as nonlinear predictive sensitivity checks. Imputation, scaling where used, and model fitting are performed inside each training fold. We report out-of-sample error, rank agreement, and sign accuracy. Coefficient summaries are reported only for ridge models and are descriptive, not hypothesis tests.


In [17]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from scipy import stats
from sklearn.dummy import DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor

try:
    from xgboost import XGBRegressor
except ImportError:
    XGBRegressor = None


def _project_dir_for_notebook_handoff() -> Path:
    cwd = Path.cwd()
    candidates = [
        cwd,
        cwd.parent,
        cwd / "meta-feature-analysis",
    ]
    for candidate in candidates:
        if (candidate / "src" / "mfa").exists() and (candidate / "notebooks").exists():
            return candidate
    raise RuntimeError(
        "Run this notebook from the repository root or notebooks/ directory."
    )


MFA_HANDOFF_DIR = (
    _project_dir_for_notebook_handoff() / ".mfa_cache" / "notebook_handoff"
)
MFA_HANDOFF_METADATA_PATH = MFA_HANDOFF_DIR / "metadata.json"
MFA_HANDOFF_METADATA = (
    json.loads(MFA_HANDOFF_METADATA_PATH.read_text(encoding="utf-8"))
    if MFA_HANDOFF_METADATA_PATH.exists()
    else {}
)

if "available_context_columns" not in globals() and MFA_HANDOFF_METADATA:
    available_context_columns = set(
        MFA_HANDOFF_METADATA.get("available_context_columns", [])
    )

PREDICTIVE_TARGET = globals().get(
    "ASSOCIATION_TARGET", MFA_HANDOFF_METADATA.get("target", "delta_norm")
)
PREDICTIVE_UNIT_COLUMN = globals().get(
    "INDEPENDENT_UNIT_COLUMN", MFA_HANDOFF_METADATA.get("unit_column", "dataset")
)
PREDICTIVE_MIN_N = globals().get("ASSOCIATION_MIN_N", 30)
PREDICTIVE_RIDGE_ALPHAS = np.logspace(-3, 3, 13)
PREDICTIVE_TOP_COEFFICIENTS = 20
PREDICTIVE_TREE_MAX_DEPTH = 2
PREDICTIVE_TREE_MIN_SAMPLES_LEAF = 5
PREDICTIVE_RANDOM_STATE = 20260424

PREDICTIVE_GENERAL_CONTROL_CANDIDATES = [
    ("n_samples", ("n_samples", "n_instances", "n_rows", "n")),
    ("n_features", ("n_features", "n_columns", "n_attrs", "n_attr", "d")),
    (
        "feature_sample_ratio",
        ("d_over_n", "n_features_over_n_samples", "features_per_sample"),
    ),
]
PREDICTIVE_CLASSIFICATION_CONTROL_CANDIDATES = [
    ("n_classes", ("n_classes", "nr_classes")),
    (
        "class_imbalance_ratio",
        ("class_imbalance_ratio", "imbalance_ratio", "majority_minority_ratio"),
    ),
]

## Load Handoff Artifacts

If the reduced matrices are already in memory, they are used as-is. Otherwise, this notebook loads the artifacts written by the export cell in `01_mfa.ipynb`.

In [18]:
MFA_REQUIRED_HANDOFF_TABLES = {
    "analysis_general_reduced": "analysis_general_reduced.pkl",
    "analysis_classification_reduced": "analysis_classification_reduced.pkl",
}

missing_handoff_tables = [
    table_name
    for table_name in MFA_REQUIRED_HANDOFF_TABLES
    if table_name not in globals()
]

if missing_handoff_tables:
    missing_paths = [
        MFA_HANDOFF_DIR / file_name
        for table_name, file_name in MFA_REQUIRED_HANDOFF_TABLES.items()
        if table_name in missing_handoff_tables
        and not (MFA_HANDOFF_DIR / file_name).exists()
    ]
    if missing_paths:
        raise FileNotFoundError(
            "Missing MFA handoff artifacts. Run the export cell at the end of "
            f"01_mfa.ipynb first. Missing files: {missing_paths}"
        )

    for table_name in missing_handoff_tables:
        globals()[table_name] = pd.read_pickle(
            MFA_HANDOFF_DIR / MFA_REQUIRED_HANDOFF_TABLES[table_name]
        )
    print(f"Loaded MFA handoff artifacts from {MFA_HANDOFF_DIR}")
else:
    print("Using reduced analysis matrices already present in memory.")

handoff_summary = []
for table_name in MFA_REQUIRED_HANDOFF_TABLES:
    table = globals()[table_name]
    handoff_summary.append(
        {
            "table": table_name,
            "rows": int(len(table)),
            "columns": int(table.shape[1]),
            "unique_units": (
                int(table[PREDICTIVE_UNIT_COLUMN].nunique(dropna=False))
                if PREDICTIVE_UNIT_COLUMN in table.columns
                else None
            ),
        }
    )

display(pd.DataFrame(handoff_summary))

Using reduced analysis matrices already present in memory.


,table,rows,columns,unique_units
0,analysis_general_reduced,51,303,51
1,analysis_classification_reduced,38,575,38


In [19]:
def _predictive_context_columns() -> set[str]:
    context = set(globals().get("available_context_columns", []))
    context.update(
        {
            PREDICTIVE_UNIT_COLUMN,
            PREDICTIVE_TARGET,
            "comparison_name",
            "task_type",
            "dataset",
        }
    )
    return context


def _assert_dataset_level_for_prediction(
    table: pd.DataFrame,
    *,
    table_name: str,
) -> dict[str, int | str]:
    if PREDICTIVE_UNIT_COLUMN not in table.columns:
        raise KeyError(f"{table_name} is missing {PREDICTIVE_UNIT_COLUMN!r}.")
    n_rows = len(table)
    n_units = table[PREDICTIVE_UNIT_COLUMN].nunique(dropna=False)
    duplicate_mask = table[PREDICTIVE_UNIT_COLUMN].duplicated(keep=False)
    if duplicate_mask.any():
        duplicate_units = (
            table.loc[duplicate_mask, PREDICTIVE_UNIT_COLUMN]
            .drop_duplicates()
            .head(10)
            .tolist()
        )
        raise ValueError(
            f"{table_name} is not dataset-level: {n_rows} rows but {n_units} unique "
            f"{PREDICTIVE_UNIT_COLUMN} values. Repeated examples: {duplicate_units}."
        )
    return {
        "analysis_table": table_name,
        "independent_unit": PREDICTIVE_UNIT_COLUMN,
        "rows": n_rows,
        "unique_units": n_units,
    }


def _numeric_usable(series: pd.Series, *, min_n: int = 3) -> bool:
    values = pd.to_numeric(series, errors="coerce").replace([np.inf, -np.inf], np.nan)
    values = values.dropna()
    return len(values) >= min_n and values.nunique() > 1


def _feature_columns_for_prediction(table: pd.DataFrame) -> list[str]:
    context = _predictive_context_columns()
    columns = []
    for column in table.columns:
        if column in context:
            continue
        if _numeric_usable(table[column]):
            columns.append(column)
    return columns


def _available_controls(
    table: pd.DataFrame,
    *,
    include_classification_controls: bool,
) -> tuple[list[str], pd.DataFrame]:
    candidates = list(PREDICTIVE_GENERAL_CONTROL_CANDIDATES)
    if include_classification_controls:
        candidates.extend(PREDICTIVE_CLASSIFICATION_CONTROL_CANDIDATES)

    selected = []
    rows = []
    for control_name, aliases in candidates:
        present_alias = next(
            (
                alias
                for alias in aliases
                if alias in table.columns and _numeric_usable(table[alias])
            ),
            None,
        )
        if present_alias is not None:
            selected.append(present_alias)
        rows.append(
            {
                "control": control_name,
                "selected_column": present_alias,
                "available": present_alias is not None,
            }
        )
    return selected, pd.DataFrame(rows)


def _ridge_pipeline() -> Pipeline:
    return Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
            ("scaler", StandardScaler()),
            ("ridge", RidgeCV(alphas=PREDICTIVE_RIDGE_ALPHAS, cv=None)),
        ]
    )


def _tree_pipeline() -> Pipeline:
    return Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
            (
                "tree",
                DecisionTreeRegressor(
                    max_depth=PREDICTIVE_TREE_MAX_DEPTH,
                    min_samples_leaf=PREDICTIVE_TREE_MIN_SAMPLES_LEAF,
                    random_state=PREDICTIVE_RANDOM_STATE,
                ),
            ),
        ]
    )


def _xgboost_pipeline() -> Pipeline | None:
    if XGBRegressor is None:
        return None
    return Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
            (
                "xgboost",
                XGBRegressor(
                    objective="reg:squarederror",
                    n_estimators=50,
                    max_depth=1,
                    learning_rate=0.05,
                    subsample=0.80,
                    colsample_bytree=0.80,
                    min_child_weight=5,
                    reg_alpha=1.0,
                    reg_lambda=10.0,
                    random_state=PREDICTIVE_RANDOM_STATE,
                    n_jobs=1,
                    verbosity=0,
                ),
            ),
        ]
    )


def _estimator_for_model(model_family: str) -> Pipeline | DummyRegressor | None:
    if model_family == "mean":
        return DummyRegressor(strategy="mean")
    if model_family == "ridge":
        return _ridge_pipeline()
    if model_family == "tree":
        return _tree_pipeline()
    if model_family == "xgboost":
        return _xgboost_pipeline()
    raise ValueError(f"Unknown model family: {model_family!r}.")


def _model_predictions(
    data: pd.DataFrame,
    predictors: list[str],
    *,
    model_name: str,
    model_family: str,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    y = data[PREDICTIVE_TARGET].astype(float).to_numpy()
    prediction_rows = []
    coefficient_rows = []
    estimator_template = _estimator_for_model(model_family)

    if estimator_template is None:
        return pd.DataFrame(), pd.DataFrame()

    for test_idx in range(len(data)):
        train_idx = np.array([idx for idx in range(len(data)) if idx != test_idx])
        train = data.iloc[train_idx]
        test = data.iloc[[test_idx]]
        y_train = y[train_idx]

        if predictors:
            estimator = _estimator_for_model(model_family)
            estimator.fit(train[predictors], y_train)
            y_pred = float(estimator.predict(test[predictors])[0])
            alpha = np.nan
            if model_family == "ridge":
                alpha = float(estimator.named_steps["ridge"].alpha_)
                coefficients = estimator.named_steps["ridge"].coef_
                for predictor, coefficient in zip(
                    predictors, coefficients, strict=True
                ):
                    coefficient_rows.append(
                        {
                            "model": model_name,
                            "held_out_dataset": test[PREDICTIVE_UNIT_COLUMN].iloc[0],
                            "predictor": predictor,
                            "coefficient": float(coefficient),
                            "alpha": alpha,
                        }
                    )
        else:
            estimator = _estimator_for_model("mean")
            estimator.fit(np.zeros((len(train), 1)), y_train)
            y_pred = float(estimator.predict(np.zeros((1, 1)))[0])
            alpha = np.nan

        prediction_rows.append(
            {
                "model": model_name,
                "model_family": model_family,
                "dataset": test[PREDICTIVE_UNIT_COLUMN].iloc[0],
                "y_true": float(y[test_idx]),
                "y_pred": y_pred,
                "alpha": alpha,
            }
        )

    return pd.DataFrame(prediction_rows), pd.DataFrame(coefficient_rows)


def _prediction_metrics(predictions: pd.DataFrame) -> dict[str, float | int]:
    if predictions.empty:
        return {
            "n": 0,
            "mae": np.nan,
            "rmse": np.nan,
            "oos_r2": np.nan,
            "spearman_pred_obs": np.nan,
            "sign_accuracy": np.nan,
            "median_alpha": np.nan,
        }

    y_true = predictions["y_true"].astype(float).to_numpy()
    y_pred = predictions["y_pred"].astype(float).to_numpy()
    sse = float(np.sum((y_true - y_pred) ** 2))
    sst = float(np.sum((y_true - y_true.mean()) ** 2))
    nonzero_mask = y_true != 0

    if np.unique(y_true).size > 1 and np.unique(y_pred).size > 1:
        spearman = float(stats.spearmanr(y_true, y_pred).statistic)
    else:
        spearman = np.nan

    return {
        "n": int(len(predictions)),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": float(mean_squared_error(y_true, y_pred) ** 0.5),
        "oos_r2": float(1 - sse / sst) if sst > 0 else np.nan,
        "spearman_pred_obs": spearman,
        "sign_accuracy": (
            float(
                np.mean(np.sign(y_true[nonzero_mask]) == np.sign(y_pred[nonzero_mask]))
            )
            if nonzero_mask.any()
            else np.nan
        ),
        "median_alpha": (
            float(predictions["alpha"].dropna().median())
            if predictions["alpha"].notna().any()
            else np.nan
        ),
    }


def _coefficient_summary(
    coefficients: pd.DataFrame,
    *,
    feature_columns: list[str],
    top_n: int = PREDICTIVE_TOP_COEFFICIENTS,
) -> pd.DataFrame:
    if coefficients.empty:
        return pd.DataFrame()
    feature_set = set(feature_columns)
    coef = coefficients[coefficients["predictor"].isin(feature_set)].copy()
    if coef.empty:
        return pd.DataFrame()
    observed_direction = np.sign(coef["coefficient"])
    coef["nonzero_direction"] = observed_direction.where(
        observed_direction != 0, np.nan
    )
    summary = (
        coef.groupby(["model", "predictor"], as_index=False)
        .agg(
            median_standardized_coefficient=("coefficient", "median"),
            median_abs_standardized_coefficient=(
                "coefficient",
                lambda values: float(np.median(np.abs(values))),
            ),
            q05_standardized_coefficient=(
                "coefficient",
                lambda values: float(np.quantile(values, 0.05)),
            ),
            q95_standardized_coefficient=(
                "coefficient",
                lambda values: float(np.quantile(values, 0.95)),
            ),
            positive_fold_share=(
                "coefficient",
                lambda values: float(np.mean(np.asarray(values) > 0)),
            ),
            negative_fold_share=(
                "coefficient",
                lambda values: float(np.mean(np.asarray(values) < 0)),
            ),
        )
        .sort_values(
            ["model", "median_abs_standardized_coefficient", "predictor"],
            ascending=[True, False, True],
        )
    )
    summary["coefficient_sign_consistency"] = summary[
        ["positive_fold_share", "negative_fold_share"]
    ].max(axis=1)
    return summary.groupby("model", group_keys=False).head(top_n).reset_index(drop=True)


def run_predictive_meta_modeling(
    table: pd.DataFrame,
    *,
    table_name: str,
    include_classification_controls: bool,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    guard = pd.DataFrame(
        [_assert_dataset_level_for_prediction(table, table_name=table_name)]
    )
    if PREDICTIVE_TARGET not in table.columns:
        raise KeyError(f"{table_name} is missing target column {PREDICTIVE_TARGET!r}.")

    feature_columns = _feature_columns_for_prediction(table)
    control_columns, control_report = _available_controls(
        table,
        include_classification_controls=include_classification_controls,
    )
    feature_columns = [
        column for column in feature_columns if column not in set(control_columns)
    ]
    control_report.insert(0, "analysis_table", table_name)

    modeling_columns = [
        PREDICTIVE_UNIT_COLUMN,
        PREDICTIVE_TARGET,
        *control_columns,
        *feature_columns,
    ]
    data = table.loc[:, list(dict.fromkeys(modeling_columns))].copy()
    for column in data.columns:
        if column == PREDICTIVE_UNIT_COLUMN:
            continue
        data[column] = pd.to_numeric(data[column], errors="coerce")
    data = data.replace([np.inf, -np.inf], np.nan)
    data = data[data[PREDICTIVE_TARGET].notna()].reset_index(drop=True)

    model_specs = [("mean_baseline", "mean", [])]
    if control_columns:
        model_specs.append(("ridge_controls", "ridge", control_columns))
    model_specs.append(("ridge_meta_features", "ridge", feature_columns))
    if control_columns:
        model_specs.append(
            (
                "ridge_controls_plus_meta_features",
                "ridge",
                control_columns + feature_columns,
            )
        )
    model_specs.append(("decision_tree_meta_features", "tree", feature_columns))
    if control_columns:
        model_specs.append(
            (
                "decision_tree_controls_plus_meta_features",
                "tree",
                control_columns + feature_columns,
            )
        )
    if XGBRegressor is not None:
        model_specs.append(("xgboost_meta_features", "xgboost", feature_columns))
        if control_columns:
            model_specs.append(
                (
                    "xgboost_controls_plus_meta_features",
                    "xgboost",
                    control_columns + feature_columns,
                )
            )

    prediction_frames = []
    coefficient_frames = []
    metric_rows = []
    for model_name, model_family, predictors in model_specs:
        if len(data) < PREDICTIVE_MIN_N:
            predictions = pd.DataFrame()
            coefficients = pd.DataFrame()
        else:
            predictions, coefficients = _model_predictions(
                data,
                predictors,
                model_name=model_name,
                model_family=model_family,
            )
        predictions.insert(0, "analysis_table", table_name)
        if not coefficients.empty:
            coefficients.insert(0, "analysis_table", table_name)
        prediction_frames.append(predictions)
        coefficient_frames.append(coefficients)
        row = {
            "analysis_table": table_name,
            "model": model_name,
            "model_family": model_family,
            "n_predictors": len(predictors),
        }
        row.update(_prediction_metrics(predictions))
        metric_rows.append(row)

    metrics = pd.DataFrame(metric_rows)
    if "mean_baseline" in set(metrics["model"]):
        baseline = metrics.set_index("model").loc["mean_baseline"]
        metrics["delta_mae_vs_baseline"] = baseline["mae"] - metrics["mae"]
        metrics["delta_rmse_vs_baseline"] = baseline["rmse"] - metrics["rmse"]

    predictions = pd.concat(prediction_frames, ignore_index=True)
    coefficients = (
        pd.concat(
            [frame for frame in coefficient_frames if not frame.empty],
            ignore_index=True,
        )
        if any(not frame.empty for frame in coefficient_frames)
        else pd.DataFrame()
    )
    coefficient_summary = _coefficient_summary(
        coefficients, feature_columns=feature_columns
    )
    if not coefficient_summary.empty:
        coefficient_summary.insert(0, "analysis_table", table_name)

    guard["n_modeling_rows"] = len(data)
    guard["n_feature_predictors"] = len(feature_columns)
    guard["n_control_predictors"] = len(control_columns)
    guard["xgboost_available"] = XGBRegressor is not None
    return metrics, predictions, coefficient_summary, control_report, guard

In [20]:
(
    general_predictive_metrics,
    general_predictive_predictions,
    general_predictive_coefficients,
    general_predictive_controls,
    general_predictive_guard,
) = run_predictive_meta_modeling(
    analysis_general_reduced,
    table_name="regression_classification",
    include_classification_controls=False,
)

(
    classification_predictive_metrics,
    classification_predictive_predictions,
    classification_predictive_coefficients,
    classification_predictive_controls,
    classification_predictive_guard,
) = run_predictive_meta_modeling(
    analysis_classification_reduced,
    table_name="classification_augmented",
    include_classification_controls=True,
)


def _concat_existing(frames: list[pd.DataFrame]) -> pd.DataFrame:
    nonempty = [frame for frame in frames if not frame.empty]
    return pd.concat(nonempty, ignore_index=True) if nonempty else pd.DataFrame()


predictive_metrics = _concat_existing(
    [general_predictive_metrics, classification_predictive_metrics]
)
predictive_predictions = _concat_existing(
    [general_predictive_predictions, classification_predictive_predictions]
)
predictive_coefficients = _concat_existing(
    [general_predictive_coefficients, classification_predictive_coefficients]
)
predictive_controls = _concat_existing(
    [general_predictive_controls, classification_predictive_controls]
)
predictive_guard = _concat_existing(
    [general_predictive_guard, classification_predictive_guard]
)

display(predictive_guard)
display(predictive_metrics)
display(predictive_controls)
display(predictive_coefficients)

,analysis_table,independent_unit,rows,unique_units,n_modeling_rows,n_feature_predictors,n_control_predictors,xgboost_available
0,regression_classification,dataset,51,51,51,283,1,True
1,classification_augmented,dataset,38,38,38,553,3,True


,analysis_table,model,model_family,n_predictors,n,mae,rmse,oos_r2,spearman_pred_obs,sign_accuracy,median_alpha,delta_mae_vs_baseline,delta_rmse_vs_baseline
0,regression_classification,mean_baseline,mean,0,51,0.240973,0.306440,-4.040000e-02,-1.000000,0.784314,NaN,0.000000,0.000000
1,regression_classification,ridge_controls,ridge,1,51,0.241892,0.307387,-4.683978e-02,-0.873575,0.784314,31.622777,-0.000919,-0.000947
2,regression_classification,ridge_meta_features,ridge,283,51,223.328955,1588.728726,-2.796454e+07,0.215566,0.764706,1000.000000,-223.087982,-1588.422286
3,regression_classification,ridge_controls_plus_meta_features,ridge,284,51,223.647223,1591.003995,-2.804470e+07,0.215566,0.764706,1000.000000,-223.406249,-1590.697554
4,regression_classification,decision_tree_meta_features,tree,283,51,0.248000,0.324121,-1.639194e-01,-0.035747,0.745098,NaN,-0.007026,-0.017681
5,regression_classification,decision_tree_controls_plus_meta_features,tree,284,51,0.247771,0.323824,-1.617858e-01,-0.035747,0.745098,NaN,-0.006798,-0.017383
6,regression_classification,xgboost_meta_features,xgboost,283,51,0.233874,0.300005,2.841516e-03,0.192489,0.784314,NaN,0.007100,0.006436
7,regression_classification,xgboost_controls_plus_meta_features,xgboost,284,51,0.235483,0.301717,-8.576836e-03,0.165430,0.784314,NaN,0.005491,0.004723
8,classification_augmented,mean_baseline,mean,0,38,0.275738,0.340199,-5.478451e-02,-1.000000,0.736842,NaN,0.000000,0.000000
9,classification_augmented,ridge_controls,ridge,3,38,0.279023,0.343550,-7.566894e-02,-0.671518,0.736842,100.000000,-0.003286,-0.003351


,analysis_table,control,selected_column,available
0,regression_classification,n_samples,None,False
1,regression_classification,n_features,None,False
2,regression_classification,feature_sample_ratio,d_over_n,True
3,classification_augmented,n_samples,None,False
4,classification_augmented,n_features,None,False
5,classification_augmented,feature_sample_ratio,d_over_n,True
6,classification_augmented,n_classes,n_classes,True
7,classification_augmented,class_imbalance_ratio,class_imbalance_ratio,True


,analysis_table,model,predictor,median_standardized_coefficient,median_abs_standardized_coefficient,q05_standardized_coefficient,q95_standardized_coefficient,positive_fold_share,negative_fold_share,coefficient_sign_consistency
0,regression_classification,ridge_controls_plus_meta_features,pymfe__attr_ent.max,-0.004286,0.004286,-0.009182,-0.004135,0.000000,1.000000,1.000000
1,regression_classification,ridge_controls_plus_meta_features,pymfe__attr_ent.skewness,-0.004194,0.004194,-0.007846,-0.003909,0.000000,1.000000,1.000000
2,regression_classification,ridge_controls_plus_meta_features,pymfe__attr_conc.iq_range,-0.003979,0.003979,-0.008817,-0.003828,0.019608,0.980392,0.980392
3,regression_classification,ridge_controls_plus_meta_features,pymfe__attr_ent.sd,-0.003930,0.003930,-0.008240,-0.003563,0.000000,1.000000,1.000000
4,regression_classification,ridge_controls_plus_meta_features,log_n,-0.003927,0.003927,-0.009241,-0.003467,0.000000,1.000000,1.000000
...,...,...,...,...,...,...,...,...,...,...
75,classification_augmented,ridge_meta_features,pymfe__attr_conc.sd,-0.002710,0.002710,-0.005996,-0.002397,0.000000,1.000000,1.000000
76,classification_augmented,ridge_meta_features,pymfe__naive_bayes.iq_range,0.002664,0.002664,0.002479,0.004979,1.000000,0.000000,1.000000
77,classification_augmented,ridge_meta_features,pymfe__attr_ent.skewness,-0.002619,0.002619,-0.004891,-0.002381,0.000000,1.000000,1.000000
78,classification_augmented,ridge_meta_features,pymfe__elite_nn.histogram.7,0.002509,0.002509,0.002242,0.004902,1.000000,0.000000,1.000000


## Reporting Rule

The predictive claim is limited to out-of-sample performance relative to the mean-only baseline. Ridge regression remains the primary predictive model. The shallow decision tree and optional strongly regularized XGBoost model are nonlinear sensitivity checks; they are useful only if they improve MAE/RMSE under leave-one-dataset-out evaluation on the corresponding dataset-level matrix. Ridge coefficient tables are descriptive only and are not interpreted as causal effects or post-selection significance tests.
